# Zephyr taxonomic profiling — analysis

Narrative write-up built on the pipeline outputs in `results/`.
Run the pipeline first:

```bash
python scripts/run_batch.py --input data/ --config config.yaml
```

Then execute this notebook for the two headline questions.

In [ ]:
import os, pandas as pd
from IPython.display import Image, display
RES = 'results'
qc = pd.read_csv(os.path.join(RES, 'qc.tsv'), sep='\t')
tax = pd.read_csv(os.path.join(RES, 'detections_by_taxon.tsv'), sep='\t')
contig = pd.read_csv(os.path.join(RES, 'detections_by_contig.tsv'), sep='\t')
print(f'{qc.shape[0]} pools, {qc.n_reads.sum():,} reads total')
qc

## Q1 — What viruses are present, and how confident?

In [ ]:
order = {'HIGH': 0, 'MEDIUM': 1, 'LOW': 2}
view = tax.sort_values(['confidence', 'aln_reads'],
                       key=lambda s: s.map(order) if s.name == 'confidence' else -s)
display(view)
print('\nDetections per confidence tier:')
print(tax.confidence.value_counts())
display(Image(os.path.join(RES, 'figures', 'detection_heatmap.png')))

## Q2 — Genome coverage for common respiratory viruses

`breadth` = fraction of genome covered; `mean_depth`/`evenness_cv` = how well.
Influenza is per-segment. Below: the best-covered detections and their coverage plots.

In [ ]:
cov = (contig[contig.breadth > 0]
       .sort_values('breadth', ascending=False)
       [['pool','taxon','taxon_group','aln_reads','breadth','mean_depth','max_depth','evenness_cv','confidence']])
display(cov.head(20))
covdir = os.path.join(RES, 'figures', 'coverage')
for f in sorted(os.listdir(covdir))[:8] if os.path.isdir(covdir) else []:
    display(Image(os.path.join(covdir, f)))